# SealedRead trên Colab

Chạy lần lượt từ trên xuống. Chi tiết: `docs/05-running.md`.

1. Chọn **Runtime → Change runtime type → A100 GPU**.
2. Các bước dài chạy nền và ghi log ra `/content/logs/`; ô cuối notebook để xem log.
3. Kết quả được chép sang Google Drive (`MyDrive/sealedread/`) mỗi 5 phút, nên runtime bị reset vẫn chạy tiếp được (marker trên Drive).
4. Thứ tự: **diễn tập nhỏ (Qwen3-0.6B)** trước, rồi mới tới dữ liệu thật và S0 với 4B. Phần huấn luyện lớn chạy trên máy 8×B200 (`scripts/cluster/`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Drive: shard gốc (dùng chung với ragproj2 / keychain-dv) và nơi lưu kết quả
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Lấy code. Repo private: tạo fine-grained token chỉ đọc (Contents: read) cho repo này,
# lưu vào Colab Secrets với tên GITHUB_TOKEN.
import os, subprocess
REPO_URL = 'github.com/<owner>/sealedread.git'   # sửa <owner> cho đúng
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'
if not os.path.exists('/content/sealedread'):
    subprocess.run(['git', 'clone', '-q', url, '/content/sealedread'], check=True)
else:
    subprocess.run(['git', '-C', '/content/sealedread', 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', '/content/sealedread', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

In [ ]:
# Môi trường (vLLM + torch + transformers trong một venv), vài phút; rồi chạy test CPU để chắc code không vỡ
!cd /content/sealedread && bash scripts/colab/setup.sh 2>&1 | tail -5
!cd /content/sealedread && /content/venv-srd/bin/python -m pytest -q tests 2>&1 | tail -3

In [ ]:
# Khôi phục dữ liệu/kết quả cũ từ Drive (nếu có) và bật đồng bộ lên Drive mỗi 5 phút
!cd /content/sealedread && bash scripts/colab/restore.sh
!cd /content/sealedread && nohup bash scripts/colab/persist.sh 300 > /content/logs/persist.log 2>&1 &

In [ ]:
# Hàm tiện ích: chạy một lệnh nền với log riêng, và xem log
import subprocess, pathlib
def bg(name, cmd):
    log = open(f'/content/logs/{name}.log', 'wb')
    p = subprocess.Popen(['bash', '-lc', f'cd /content/sealedread && {cmd}'], stdout=log, stderr=subprocess.STDOUT,
                         start_new_session=True)
    print('started', name, 'pid', p.pid)
def tail(name, n=30):
    p = pathlib.Path(f'/content/logs/{name}.log')
    print('\n'.join(p.read_text(errors='replace').splitlines()[-n:]) if p.exists() else '(no log yet)')

In [ ]:
# Bước 1: diễn tập toàn bộ chuỗi với Qwen3-0.6B, ngữ cảnh 4K-16K, vài trăm câu (vài giờ A100).
# Con số không có ý nghĩa; mọi script phải chạy hết không cần sửa tay. Teacher ở đây là chính 0.6B.
bg('rehearsal', 'bash scripts/colab/run_rehearsal.sh')

In [ ]:
# Bước 2: dữ liệu thật (M1): shard -> tokenizer check -> build (đếm token bằng tokenizer Qwen3) -> kiểm tra dữ liệu
bg('data', 'bash scripts/colab/run_data.sh')

In [ ]:
# Bước 3: S0 trên 4B: closed-book (kiểm tra 2), paraphrase, teacher pilot (caps), parity (kiểm tra 6), đo chi phí,
# 6 cấu hình P0 và quyết định của bảng 6. Cần dữ liệu ở bước 2. Teacher 27B cần GPU đủ lớn (xem configs/teacher.yaml).
bg('s0', 'bash scripts/colab/run_s0.sh')

In [ ]:
# Một lần SFT (chạy lại đúng lệnh để tiếp tục sau khi runtime reset)
# bg('sft_sr', 'bash scripts/colab/run_sft.sh configs/sft.yaml train.branch=sealedread run.name=sft_sealedread_s0')

In [ ]:
tail('rehearsal')